# Uh ROI 引导的 Udeco 边沿实验

本实验严格配对 Uh2→Udeco2、Uh3→Udeco3。Uh 只确定事件 ROI；Udeco 在完整 ROI 内独立检测候选，再回到原信号定位、计算形态并各选最大幅度 rise/fall。图表均可缩放、平移。每个阶段代码 Cell 顶部都有 local override；改动后只需重跑该 Cell 及后续 Cell。

In [ ]:
# Cell 2：全局默认值、依赖和通道语义
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import clear_output, display

from udeco_uh_roi_edge_detection import (
    ExperimentConfig, baseline_sigmas, build_event_table,
    detect_roi_candidates, find_uh_rois, load_paired_sample,
    localize_roi_candidates, preprocess_channel,
)

pio.renderers.default = "plotly_mimetype"
SAMPLE_NAME = "cgx_rope_1"
DATA_ROOT = Path("subject_data")
FS = 100
BASELINE_SEC = 15.0
MEDIAN_KERNEL = 4
CONTEXT_WINDOW_SEC = 10.0
CONTEXT_ROBUST_ITERS = 1

UH_PEAK_PROMINENCE_SIGMA = 4.0
UH_PEAK_TO_COARSE_L_SIGMA = 10.0  # Nbar：peak - coarse L 必须严格大于 Nbar × 前15 s sigma_uh
UH_PEAK_DISTANCE_SEC = 0.30
UH_FLAT_SLOPE_SIGMA = 2.0
UH_FLAT_MIN_SEC = 0.05
UH_ROI_MARGIN_SEC = 0.15

EDGE_DETECTOR = "median_diff"  # 或 "haar"；二者不混用、不自动 fallback
MEDIAN_DIFF_WINDOWS = (1, 3, 5, 7)
EDGE_SCORE_SIGMA = 3.5
CANDIDATE_MERGE_SEC = 0.02
HAAR_SCALES_SEC = (0.02, 0.04, 0.08, 0.16, 0.32, 0.64, 1.28, 2.56)
HAAR_SINGLE_SCALE_SIGMA = 3.5
HAAR_DERIVATIVE_SIGMA = 3.5
HAAR_MIN_SCALE_SUPPORT = 2
HAAR_MERGE_SEC = 0.08
HAAR_FRAGMENT_MERGE_SEC = 0.02
HAAR_FRAGMENT_MIN_SPEARMAN = 0.50
HAAR_FRAGMENT_MIN_FORWARD_FRACTION = 0.60

LOCALIZE_SEARCH_SEC = 0.20
PRE_PLATEAU_SEC = 0.05
POST_PLATEAU_SEC = 0.05
PLATEAU_SIGMA_K = 2.5
DERIV_CORE_RATIO = 0.20
DERIV_SIGMA_K = 3.0
CORE_END_MIN_POINTS = 2
SETTLE_WINDOW_SEC = 0.07
SETTLE_DERIV_SIGMA_K = 2.5
SETTLE_HOLD_SEC = 0.05

MIN_AMPLITUDE_SIGMA_K = 3.0
SHARP_CORE_DURATION_SEC = 0.05
DERIV_PEAK_WIDTH_THRESHOLD = 0.04
DERIV_PEAK_SIGMA_K = 5.0
LONG_TAIL_SEC = 0.10
DIRECTION_MAP = {
    2: {1: {"sharp": "detach", "gradual": "release"},
        -1: {"sharp": "attach", "gradual": "pressure"}},
    3: {1: {"sharp": "detach", "gradual": "release"},
        -1: {"sharp": "attach", "gradual": "pressure"}},
}
PLOT_START_SEC = None
PLOT_END_SEC = None
BASE_CONFIG = ExperimentConfig(
    median_kernel=MEDIAN_KERNEL, context_window_sec=CONTEXT_WINDOW_SEC,
    context_robust_iters=CONTEXT_ROBUST_ITERS, baseline_sec=BASELINE_SEC,
    uh_peak_prominence_sigma=UH_PEAK_PROMINENCE_SIGMA,
    uh_peak_to_coarse_l_sigma=UH_PEAK_TO_COARSE_L_SIGMA,
    uh_peak_distance_sec=UH_PEAK_DISTANCE_SEC,
    uh_flat_slope_sigma=UH_FLAT_SLOPE_SIGMA, uh_flat_min_sec=UH_FLAT_MIN_SEC,
    uh_roi_margin_sec=UH_ROI_MARGIN_SEC, edge_detector=EDGE_DETECTOR,
    median_diff_windows=MEDIAN_DIFF_WINDOWS, edge_score_sigma=EDGE_SCORE_SIGMA,
    candidate_merge_sec=CANDIDATE_MERGE_SEC, haar_scales_sec=HAAR_SCALES_SEC,
    haar_single_scale_sigma=HAAR_SINGLE_SCALE_SIGMA,
    haar_derivative_sigma=HAAR_DERIVATIVE_SIGMA,
    haar_min_scale_support=HAAR_MIN_SCALE_SUPPORT,
    haar_merge_sec=HAAR_MERGE_SEC,
    haar_fragment_merge_sec=HAAR_FRAGMENT_MERGE_SEC,
    haar_fragment_min_spearman=HAAR_FRAGMENT_MIN_SPEARMAN,
    haar_fragment_min_forward_fraction=HAAR_FRAGMENT_MIN_FORWARD_FRACTION,
    localize_search_sec=LOCALIZE_SEARCH_SEC, pre_plateau_sec=PRE_PLATEAU_SEC,
    post_plateau_sec=POST_PLATEAU_SEC, plateau_sigma_k=PLATEAU_SIGMA_K,
    deriv_core_ratio=DERIV_CORE_RATIO, deriv_sigma_k=DERIV_SIGMA_K,
    core_end_min_points=CORE_END_MIN_POINTS, settle_window_sec=SETTLE_WINDOW_SEC,
    settle_deriv_sigma_k=SETTLE_DERIV_SIGMA_K, settle_hold_sec=SETTLE_HOLD_SEC,
    min_amplitude_sigma_k=MIN_AMPLITUDE_SIGMA_K,
    sharp_core_duration_sec=SHARP_CORE_DURATION_SEC,
    deriv_peak_width_threshold_sec=DERIV_PEAK_WIDTH_THRESHOLD,
    deriv_peak_sigma_k=DERIV_PEAK_SIGMA_K, long_tail_sec=LONG_TAIL_SEC,
)
print("Configured", SAMPLE_NAME, "at", FS, "Hz; detector:", EDGE_DETECTOR)

## 1. 数据读取与四路 raw signals

使用现有 Pydisplay 序号修复和 Udeco 标定。以下四图共用时间 x 轴，灰色区域为前 15 s 静止佩戴基线。

In [ ]:
# Local override：数据与初始显示范围
sample_name = SAMPLE_NAME
fs = FS
data_root = DATA_ROOT
plot_start_sec = PLOT_START_SEC
plot_end_sec = PLOT_END_SEC

matches = sorted(data_root.glob(f"*/{sample_name}_sensor.csv"))
if len(matches) != 1:
    raise FileNotFoundError(f"Expected one sensor CSV for {sample_name}, found {matches}")
sample = load_paired_sample(matches[0], data_root=data_root, fs=fs)
time_s = sample["time_s"]
fs = sample["fs"]
raw_signals = sample["signals"]
print(sample["sample_name"], sample["subject"], len(time_s), "samples")
raw_fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.045,
                        subplot_titles=list(raw_signals))
for row, (name, values) in enumerate(raw_signals.items(), start=1):
    raw_fig.add_trace(go.Scattergl(x=time_s, y=values, name=name, line=dict(width=1)), row=row, col=1)
    raw_fig.add_vrect(x0=0, x1=BASELINE_SEC, fillcolor="silver", opacity=0.15,
                      line_width=0, row=row, col=1)
    raw_fig.update_yaxes(title_text=name, row=row, col=1)
raw_fig.update_xaxes(title_text="Time (s)", row=4, col=1)
if plot_start_sec is not None and plot_end_sec is not None:
    raw_fig.update_xaxes(range=[plot_start_sec, plot_end_sec])
raw_fig.update_layout(height=1050, title=f"{sample_name}: four raw channels",
                      template="plotly_white", dragmode="pan", hovermode="x unified")
raw_fig.show(config={"scrollZoom": True, "displaylogo": False})

## 2. 4 点 median、Bcontext 与固定基线

四路都按 raw → clean → Bcontext → clean−Bcontext 处理。后续所有噪声阈值由前 15 s 给出，绝不在事件附近重估计。

In [ ]:
# Local override：预处理和基线
median_kernel = MEDIAN_KERNEL
context_window_sec = CONTEXT_WINDOW_SEC
context_robust_iters = CONTEXT_ROBUST_ITERS
baseline_sec = BASELINE_SEC

preprocess_config = replace(BASE_CONFIG, median_kernel=median_kernel,
                            context_window_sec=context_window_sec,
                            context_robust_iters=context_robust_iters,
                            baseline_sec=baseline_sec)
processed = {name: preprocess_channel(values, fs, preprocess_config)
             for name, values in raw_signals.items()}
noise = {channel: baseline_sigmas(processed[f"Uh{channel}"]["edge_signal"],
                                  processed[f"Udeco{channel}"]["edge_signal"],
                                  fs=fs, baseline_sec=baseline_sec)
         for channel in (2, 3)}
for channel, n in noise.items():
    print(f"Pair {channel}: sigma_uh={n['sigma_uh']:.6g}, sigma_udeco={n['sigma_udeco']:.6g}, "
          f"sigma_du={n['sigma_du']:.6g}, sigma_duh={n['sigma_duh']:.6g} (first {baseline_sec:g} s)")

def plot_preprocessed(name):
    values = processed[name]
    keys = ("raw", "clean", "Bcontext", "edge_signal")
    fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.04,
                        subplot_titles=("raw", "4-point median clean", "Bcontext", "clean - Bcontext"))
    for row, key in enumerate(keys, start=1):
        fig.add_trace(go.Scattergl(x=time_s, y=values[key], name=key,
                                   line=dict(width=1)), row=row, col=1)
        fig.add_vrect(x0=0, x1=baseline_sec, fillcolor="silver", opacity=0.15,
                      line_width=0, row=row, col=1)
    fig.update_xaxes(title_text="Time (s)", row=4, col=1)
    fig.update_layout(height=900, title=f"{sample_name}: {name} preprocessing",
                      template="plotly_white", dragmode="pan", hovermode="x unified")
    fig.show(config={"scrollZoom": True, "displaylogo": False})
    return fig

preprocessing_figures = {name: plot_preprocessed(name) for name in raw_signals}

## 3. Uh peaks、coarse bases、修正边界与 ROI

仅 Uh 正向局部峰参与 ROI 生成，`edge_signal[peak] - edge_signal[coarse L]` 须严格大于 `Nbar × sigma_uh`（默认 Nbar=10，sigma_uh 仅取前 15 s 基线）。prominence 仍要求峰相对周围谷底足够突出。粗基点和最终边界分开显示；重叠 ROI 合并，编号及来源峰保留。

In [ ]:
# Local override：Uh 峰与 ROI
peak_prominence_sigma = UH_PEAK_PROMINENCE_SIGMA
peak_to_coarse_l_sigma = UH_PEAK_TO_COARSE_L_SIGMA  # Nbar
peak_distance_sec = UH_PEAK_DISTANCE_SEC
flat_slope_sigma = UH_FLAT_SLOPE_SIGMA
flat_min_sec = UH_FLAT_MIN_SEC
roi_margin_sec = UH_ROI_MARGIN_SEC

roi_config = replace(preprocess_config,
                     uh_peak_prominence_sigma=peak_prominence_sigma,
                     uh_peak_to_coarse_l_sigma=peak_to_coarse_l_sigma,
                     uh_peak_distance_sec=peak_distance_sec,
                     uh_flat_slope_sigma=flat_slope_sigma,
                     uh_flat_min_sec=flat_min_sec,
                     uh_roi_margin_sec=roi_margin_sec)
roi_results = {}
for channel in (2, 3):
    edge = processed[f"Uh{channel}"]["edge_signal"]
    rois, peaks = find_uh_rois(edge, fs=fs, sigma_uh=noise[channel]["sigma_uh"],
                               sigma_duh=noise[channel]["sigma_duh"], config=roi_config)
    roi_results[channel] = {"rois": rois, "peaks": peaks}
    print(f"Uh{channel}: {len(peaks)} peaks → {len(rois)} merged ROIs")
    fig = go.Figure()
    fig.add_trace(go.Scattergl(x=time_s, y=edge, name=f"Uh{channel} edge_signal",
                               line=dict(color="#444", width=1)))
    peak_indices = [item["peak_idx"] for item in peaks]
    threshold_y = [edge[item["coarse_left_idx"]] + peak_to_coarse_l_sigma * noise[channel]["sigma_uh"] for item in peaks]
    fig.add_trace(go.Scattergl(x=time_s[peak_indices], y=threshold_y, mode="markers",
                               name=f"coarse L + {peak_to_coarse_l_sigma:g}σ_uh",
                               marker=dict(symbol="cross", color="#e41a1c", size=9)))
    for field, label, symbol, color in (
        ("peak_idx", "peak", "diamond", "#e41a1c"),
        ("coarse_left_idx", "coarse L", "circle-open", "#ff7f00"),
        ("coarse_right_idx", "coarse R", "circle-open", "#ff7f00"),
        ("refined_left_idx", "refined L", "triangle-up", "#377eb8"),
        ("refined_right_idx", "refined R", "triangle-down", "#4daf4a"),
    ):
        indices = [item[field] for item in peaks]
        fig.add_trace(go.Scattergl(x=time_s[indices], y=edge[indices], mode="markers",
                                   name=label, marker=dict(symbol=symbol, color=color, size=8),
                                   customdata=[[item["prominence"], item["coarse_left_rise"]] for item in peaks],
                                   hovertemplate="t=%{x:.3f}s; peak - coarse L=%{customdata[1]:.5g}; prominence=%{customdata[0]:.5g}<extra></extra>"))
    for roi in rois:
        x0, x1 = time_s[roi["left_idx"]], time_s[roi["right_idx"]]
        fig.add_vrect(x0=x0, x1=x1, fillcolor="#8dd3c7", opacity=0.13,
                      line_width=0, annotation_text=roi["roi_id"],
                      annotation_position="top left")
    fig.add_vrect(x0=0, x1=baseline_sec, fillcolor="silver", opacity=0.10, line_width=0)
    fig.update_layout(title=f"Uh{channel}: peak → bases → refined boundaries → ROI",
                      xaxis_title="Time (s)", yaxis_title="clean - Bcontext",
                      height=550, template="plotly_white", dragmode="pan")
    fig.show(config={"scrollZoom": True, "displaylogo": False})

## 4. ROI 内多尺度候选发现

median difference 的 `w=1` 是相邻采样差；所有尺度按前 15 s 对应 detector 响应标准差归一化。Haar 仅在切换 `EDGE_DETECTOR` 时运行。下图显示完整 Uh ROI 范围内的 Udeco 信号、各尺度响应、合成分数、阈值及全部候选区域/中心。

In [ ]:
# Local override：detector 与各尺度/fragment 参数
edge_detector = EDGE_DETECTOR
median_diff_windows = MEDIAN_DIFF_WINDOWS
edge_score_sigma = EDGE_SCORE_SIGMA
candidate_merge_sec = CANDIDATE_MERGE_SEC
haar_scales_sec = HAAR_SCALES_SEC
haar_single_scale_sigma = HAAR_SINGLE_SCALE_SIGMA
haar_derivative_sigma = HAAR_DERIVATIVE_SIGMA
haar_min_scale_support = HAAR_MIN_SCALE_SUPPORT
haar_merge_sec = HAAR_MERGE_SEC
haar_fragment_merge_sec = HAAR_FRAGMENT_MERGE_SEC
haar_fragment_min_spearman = HAAR_FRAGMENT_MIN_SPEARMAN
haar_fragment_min_forward_fraction = HAAR_FRAGMENT_MIN_FORWARD_FRACTION

detect_config = replace(roi_config, edge_detector=edge_detector,
                        median_diff_windows=tuple(median_diff_windows),
                        edge_score_sigma=edge_score_sigma,
                        candidate_merge_sec=candidate_merge_sec,
                        haar_scales_sec=tuple(haar_scales_sec),
                        haar_single_scale_sigma=haar_single_scale_sigma,
                        haar_derivative_sigma=haar_derivative_sigma,
                        haar_min_scale_support=haar_min_scale_support,
                        haar_merge_sec=haar_merge_sec,
                        haar_fragment_merge_sec=haar_fragment_merge_sec,
                        haar_fragment_min_spearman=haar_fragment_min_spearman,
                        haar_fragment_min_forward_fraction=haar_fragment_min_forward_fraction)
detections = {}
for channel in (2, 3):
    rois = roi_results[channel]["rois"]
    edge = processed[f"Udeco{channel}"]["edge_signal"]
    candidates, diagnostics = detect_roi_candidates(edge, rois, fs=fs, config=detect_config)
    detections[channel] = {"candidates": candidates, "diagnostics": diagnostics}
    print(f"Udeco{channel}: {len(candidates)} {edge_detector} candidates inside {len(rois)} Uh ROIs")
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.055,
                        subplot_titles=("Udeco edge_signal", "E_w (raw signed response)",
                                        "max_w |E_w / sigma_w|, signed winner"))
    fig.add_trace(go.Scattergl(x=time_s, y=edge, name="Udeco edge_signal",
                               line=dict(color="#444", width=1)), row=1, col=1)
    for scale, response in diagnostics["scale_responses"].items():
        fig.add_trace(go.Scattergl(x=time_s, y=response, name=f"E_{scale}",
                                   line=dict(width=1)), row=2, col=1)
    score = diagnostics["combined_score"]
    fig.add_trace(go.Scattergl(x=time_s, y=score, name="combined signed score",
                               line=dict(color="#1f78b4", width=1)), row=3, col=1)
    fig.add_hline(y=edge_score_sigma, line_dash="dash", line_color="red", row=3, col=1)
    fig.add_hline(y=-edge_score_sigma, line_dash="dash", line_color="red", row=3, col=1)
    region_x = [v for c in candidates for v in (time_s[c["region_start_idx"]],
                                                  time_s[c["region_end_idx"]], None)]
    region_y = [v for c in candidates for v in (0, 0, None)]
    fig.add_trace(go.Scatter(x=region_x, y=region_y, mode="lines", name="candidate regions",
                             line=dict(color="#ff7f00", width=5)), row=3, col=1)
    centers = [c["center_idx"] for c in candidates]
    fig.add_trace(go.Scattergl(x=time_s[centers], y=score[centers], mode="markers",
                               name="candidate centers", marker=dict(color="red", size=7)),
                  row=3, col=1)
    for roi in rois:
        fig.add_vrect(x0=time_s[roi["left_idx"]], x1=time_s[roi["right_idx"]],
                      fillcolor="#8dd3c7", opacity=0.08, line_width=0, row=1, col=1)
    fig.update_xaxes(title_text="Time (s)", row=3, col=1)
    fig.update_layout(title=f"Udeco{channel}: {edge_detector} scales and candidates",
                      height=950, template="plotly_white", dragmode="pan", hovermode="x unified")
    fig.show(config={"scrollZoom": True, "displaylogo": False})

## 5. 每个 candidate 的局部定位

检测器只给候选中心和区域；以下图回到 Udeco clean/edge_signal，显示 `start`、`core_end`、`settle_end`、导数主峰、core 阈值及 FWHM。选择器可逐个查看全部候选；未定位成功的候选显示原因，不会被选为主边沿。

In [ ]:
# Local override：局部定位、导数 core 与 settle
localize_search_sec = LOCALIZE_SEARCH_SEC
pre_plateau_sec = PRE_PLATEAU_SEC
post_plateau_sec = POST_PLATEAU_SEC
plateau_sigma_k = PLATEAU_SIGMA_K
deriv_core_ratio = DERIV_CORE_RATIO
deriv_sigma_k = DERIV_SIGMA_K
core_end_min_points = CORE_END_MIN_POINTS
settle_window_sec = SETTLE_WINDOW_SEC
settle_deriv_sigma_k = SETTLE_DERIV_SIGMA_K
settle_hold_sec = SETTLE_HOLD_SEC

localize_config = replace(detect_config, localize_search_sec=localize_search_sec,
                          pre_plateau_sec=pre_plateau_sec,
                          post_plateau_sec=post_plateau_sec,
                          plateau_sigma_k=plateau_sigma_k,
                          deriv_core_ratio=deriv_core_ratio,
                          deriv_sigma_k=deriv_sigma_k,
                          core_end_min_points=core_end_min_points,
                          settle_window_sec=settle_window_sec,
                          settle_deriv_sigma_k=settle_deriv_sigma_k,
                          settle_hold_sec=settle_hold_sec)
localized = {}
for channel in (2, 3):
    signal = processed[f"Udeco{channel}"]
    localized[channel] = localize_roi_candidates(
        signal["clean"], signal["edge_signal"], detections[channel]["candidates"],
        roi_results[channel]["rois"], fs=fs, noise=noise[channel], config=localize_config)
    print(f"Udeco{channel}: {sum(c['resolved'] for c in localized[channel])}/{len(localized[channel])} candidates localized")

def plot_candidate_local(channel, candidate_index):
    item = localized[channel][candidate_index]
    signal = processed[f"Udeco{channel}"]
    roi = next(r for r in roi_results[channel]["rois"] if r["roi_id"] == item["roi_id"])
    center = item["center_idx"]
    markers = [item.get(k) for k in ("start_idx", "core_end_idx", "settle_end_idx")]
    valid = [m for m in markers if m is not None]
    lo = max(roi["left_idx"], min([center - round(0.25 * fs), *valid]))
    hi = min(roi["right_idx"], max([center + round(0.35 * fs), *valid]))
    indices = np.arange(lo, hi + 1)
    derivative = np.diff(signal["edge_signal"], prepend=signal["edge_signal"][0])
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
                        subplot_titles=("Udeco clean / edge_signal", "diff(edge_signal) morphology"))
    fig.add_trace(go.Scatter(x=time_s[indices], y=signal["clean"][indices], name="clean"), row=1, col=1)
    fig.add_trace(go.Scatter(x=time_s[indices], y=signal["edge_signal"][indices],
                             name="edge_signal"), row=1, col=1)
    fig.add_trace(go.Scatter(x=time_s[indices], y=derivative[indices], name="dU"), row=2, col=1)
    for key, color in (("start_idx", "#377eb8"), ("core_end_idx", "#4daf4a"),
                       ("settle_end_idx", "#984ea3")):
        if item.get(key) is not None:
            fig.add_vline(x=time_s[item[key]], line_color=color, line_dash="dash",
                          annotation_text=key, row=1, col=1)
    if item.get("derivative_peak_idx") is not None:
        peak = item["derivative_peak_idx"]
        fig.add_trace(go.Scatter(x=[time_s[peak]], y=[derivative[peak]], mode="markers",
                                 name="derivative peak", marker=dict(color="red", size=11)), row=2, col=1)
    if item.get("core_threshold") is not None:
        for sign in (-1, 1):
            fig.add_hline(y=sign * item["core_threshold"], line_dash="dot",
                          line_color="red", row=2, col=1)
    if item.get("derivative_fwhm_left_idx") is not None:
        fig.add_vrect(x0=time_s[item["derivative_fwhm_left_idx"]],
                      x1=time_s[item["derivative_fwhm_right_idx"]],
                      fillcolor="orange", opacity=0.15, line_width=0, row=2, col=1)
    fig.update_xaxes(title_text="Time (s)", row=2, col=1)
    fig.update_layout(title=f"Udeco{channel} {item['roi_id']} candidate {candidate_index}: "
                            f"{'resolved' if item['resolved'] else item['reason']}",
                      template="plotly_white", height=700, dragmode="pan")
    fig.show(config={"scrollZoom": True, "displaylogo": False})
    return fig

candidate_options = [(f"Udeco{channel} {item['roi_id']} #{index} t={time_s[item['center_idx']]:.2f}s",
                      (channel, index))
                     for channel in (2, 3) for index, item in enumerate(localized[channel])]
if candidate_options:
    first_channel, first_index = candidate_options[0][1]
    first_candidate_figure = plot_candidate_local(first_channel, first_index)
    candidate_selector = widgets.Dropdown(options=candidate_options, description="Candidate:",
                                          layout=widgets.Layout(width="750px"))
    candidate_output = widgets.Output()
    def _show_selected_candidate(change):
        with candidate_output:
            clear_output(wait=True)
            channel, index = change["new"]
            plot_candidate_local(channel, index)
    candidate_selector.observe(_show_selected_candidate, names="value")
    display(candidate_selector, candidate_output)

## 6. ROI 内最大 rise/fall 与事件表

每个 ROI 最多保留一个实际幅度最大的上升沿和一个下降沿。小边缘仍在 `event_table` 中供检查，但不进入最终事件。

In [ ]:
# Local override：主边沿保护与形态分类（T_core 优先）
min_amplitude_sigma_k = MIN_AMPLITUDE_SIGMA_K
sharp_core_duration_sec = SHARP_CORE_DURATION_SEC
deriv_peak_width_threshold = DERIV_PEAK_WIDTH_THRESHOLD
deriv_peak_sigma_k = DERIV_PEAK_SIGMA_K
long_tail_sec = LONG_TAIL_SEC
direction_map = DIRECTION_MAP

classify_config = replace(localize_config, min_amplitude_sigma_k=min_amplitude_sigma_k,
                          sharp_core_duration_sec=sharp_core_duration_sec,
                          deriv_peak_width_threshold_sec=deriv_peak_width_threshold,
                          deriv_peak_sigma_k=deriv_peak_sigma_k,
                          long_tail_sec=long_tail_sec)
tables = {channel: build_event_table(localized[channel], roi_results[channel]["rois"],
                                     channel=f"Udeco{channel}",
                                     sigma_udeco=noise[channel]["sigma_udeco"],
                                     config=classify_config,
                                     direction_map=direction_map[channel])
          for channel in (2, 3)}
event_table = pd.concat(tables.values(), ignore_index=True)
for channel in (2, 3):
    table = tables[channel]
    selected = table[table.is_main_rising | table.is_main_falling]
    print(f"Udeco{channel}: {len(table)} candidates; {len(selected)} main edges")
    edge = processed[f"Udeco{channel}"]["edge_signal"]
    fig = go.Figure()
    fig.add_trace(go.Scattergl(x=time_s, y=edge, name="full Udeco edge_signal",
                               line=dict(color="#555", width=1)))
    if not table.empty:
        centers = table.center_idx.to_numpy(dtype=int)
        fig.add_trace(go.Scattergl(x=time_s[centers], y=edge[centers], mode="markers",
                                   name="all candidates", marker=dict(color="#aaaaaa", size=5)))
        for direction, name, color, symbol in ((1, "main rising", "#e41a1c", "triangle-up"),
                                                (-1, "main falling", "#377eb8", "triangle-down")):
            main = table[table.is_main_rising if direction == 1 else table.is_main_falling]
            indices = main.center_idx.to_numpy(dtype=int)
            fig.add_trace(go.Scattergl(x=time_s[indices], y=edge[indices], mode="markers",
                                       name=name, marker=dict(color=color, size=13, symbol=symbol)))
    for roi in roi_results[channel]["rois"]:
        fig.add_vrect(x0=time_s[roi["left_idx"]], x1=time_s[roi["right_idx"]],
                      fillcolor="#8dd3c7", opacity=0.07, line_width=0)
    fig.update_layout(title=f"Udeco{channel}: all candidates and max-amplitude rise/fall",
                      xaxis_title="Time (s)", yaxis_title="clean - Bcontext",
                      template="plotly_white", height=550, dragmode="pan")
    fig.show(config={"scrollZoom": True, "displaylogo": False})

def plot_roi_selection(channel, roi_id):
    roi = next(r for r in roi_results[channel]["rois"] if r["roi_id"] == roi_id)
    edge = processed[f"Udeco{channel}"]["edge_signal"]
    lo, hi = roi["left_idx"], roi["right_idx"]
    indices = np.arange(lo, hi + 1)
    rows = tables[channel].loc[tables[channel].roi_id == roi_id]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=time_s[indices], y=edge[indices], name="complete Udeco ROI"))
    for _, item in rows.iterrows():
        center = int(item.center_idx)
        selected = bool(item.is_main_rising or item.is_main_falling)
        fig.add_trace(go.Scatter(x=[time_s[center]], y=[edge[center]], mode="markers",
                                 name=("MAIN " if selected else "candidate ") + str(center),
                                 marker=dict(size=14 if selected else 6,
                                             color="#e41a1c" if item.direction > 0 else "#377eb8",
                                             opacity=1 if selected else 0.4)))
        if selected:
            for key, color in (("start_idx", "#377eb8"), ("core_end_idx", "#4daf4a"),
                               ("settle_end_idx", "#984ea3")):
                if pd.notna(item.get(key)):
                    fig.add_vline(x=time_s[int(item[key])], line_dash="dash", line_color=color,
                                  annotation_text=key)
    fig.update_layout(title=f"Udeco{channel} {roi_id}: all edges and selected rise/fall",
                      xaxis_title="Time (s)", yaxis_title="clean - Bcontext",
                      template="plotly_white", height=550, dragmode="pan")
    fig.show(config={"scrollZoom": True, "displaylogo": False})
    return fig

roi_options = [(f"Udeco{channel} {roi['roi_id']}", (channel, roi['roi_id']))
               for channel in (2, 3) for roi in roi_results[channel]["rois"]]
if roi_options:
    first_roi_figure = plot_roi_selection(*roi_options[0][1])
    roi_selector = widgets.Dropdown(options=roi_options, description="ROI:",
                                    layout=widgets.Layout(width="500px"))
    roi_output = widgets.Output()
    def _show_selected_roi(change):
        with roi_output:
            clear_output(wait=True)
            plot_roi_selection(*change["new"])
    roi_selector.observe(_show_selected_roi, names="value")
    display(roi_selector, roi_output)

## 7. Derivative morphology、分类与 event_table

`T_core` 是第一版 sharp/gradual 的优先判据；导数峰强度和 FWHM 用于观察是否存在更好的分界。短 core + 长 tail 仍是 sharp。完整 `event_table` 保留所有候选及筛选状态。

In [ ]:
# Local override：散点筛选与表格显示列
plot_main_only = True
table_columns = ["channel", "roi_id", "center_idx", "direction", "detector_name",
                 "detector_score", "resolved", "reason", "delta", "baseline_snr",
                 "start_idx", "core_end_idx", "settle_end_idx", "T_core", "T_tail",
                 "derivative_peak", "normalized_derivative_peak",
                 "derivative_peak_width", "monotonicity", "derivative_support",
                 "is_main_rising", "is_main_falling", "morphology", "event_label"]

plot_table = event_table[event_table.is_main_rising | event_table.is_main_falling] if plot_main_only else event_table
plot_table = plot_table[plot_table.resolved]
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("T_core vs normalized derivative peak",
                                    "T_core vs derivative peak FWHM"))
for channel in (2, 3):
    channel_table = plot_table[plot_table.channel == f"Udeco{channel}"]
    for morphology, group in channel_table.groupby("morphology"):
        color = "#e41a1c" if morphology.startswith("sharp") else "#377eb8"
        fig.add_trace(go.Scatter(x=group.T_core, y=group.normalized_derivative_peak,
                                 mode="markers", name=f"Udeco{channel} {morphology}",
                                 marker=dict(color=color, symbol="circle" if channel == 2 else "diamond"),
                                 text=group.roi_id, hovertemplate="%{text}<br>T_core=%{x:.3f}s<br>peak/σdu=%{y:.1f}<extra></extra>"), row=1, col=1)
        fig.add_trace(go.Scatter(x=group.T_core, y=group.derivative_peak_width,
                                 mode="markers", name=f"Udeco{channel} {morphology}",
                                 marker=dict(color=color, symbol="circle" if channel == 2 else "diamond"),
                                 showlegend=False, text=group.roi_id), row=1, col=2)
for col in (1, 2):
    fig.add_vline(x=sharp_core_duration_sec, line_dash="dash", line_color="black", row=1, col=col)
    fig.update_xaxes(title_text="T_core (s)", row=1, col=col)
fig.add_hline(y=deriv_peak_width_threshold, line_dash="dot", line_color="gray", row=1, col=2)
fig.update_yaxes(title_text="abs(derivative peak) / baseline sigma_du", row=1, col=1)
fig.update_yaxes(title_text="Derivative FWHM (s)", row=1, col=2)
fig.update_layout(title="Selected event morphology", template="plotly_white",
                  height=500, width=1150, dragmode="pan")
fig.show(config={"scrollZoom": True, "displaylogo": False})
print("Selected morphology counts:")
display(plot_table.groupby(["channel", "morphology", "event_label"]).size().rename("count").reset_index())
print("Full event_table (Pandas may truncate display; variable retains every row):")
display(event_table.reindex(columns=table_columns))

## 8. median_diff 与 Haar 对照

两个 detector 分别在同一批 Uh ROI 内运行，随后共用定位和分类；没有自动 fallback。修改上游对应 local override 后重跑本 Cell，可比较超参数的影响。

In [ ]:
# Local override：对照中的 detector 集合和显示范围
comparison_detectors = ("median_diff", "haar")
comparison_start_sec = PLOT_START_SEC
comparison_end_sec = PLOT_END_SEC

comparison_rows = []
comparison_events = {}
for detector in comparison_detectors:
    comparison_events[detector] = {}
    compare_detect_config = replace(detect_config, edge_detector=detector)
    for channel in (2, 3):
        signal = processed[f"Udeco{channel}"]
        rois = roi_results[channel]["rois"]
        candidates, _ = detect_roi_candidates(signal["edge_signal"], rois, fs=fs,
                                                config=compare_detect_config)
        measured = localize_roi_candidates(signal["clean"], signal["edge_signal"],
                                            candidates, rois, fs=fs, noise=noise[channel],
                                            config=localize_config)
        table = build_event_table(measured, rois, channel=f"Udeco{channel}",
                                  sigma_udeco=noise[channel]["sigma_udeco"],
                                  config=classify_config,
                                  direction_map=direction_map[channel])
        comparison_events[detector][channel] = table
        main = table[table.is_main_rising | table.is_main_falling]
        comparison_rows.append({"detector": detector, "channel": f"Udeco{channel}",
                                "ROI_count": len(rois), "candidates": len(table),
                                "resolved": int(table.resolved.sum()),
                                "main_rising": int(table.is_main_rising.sum()),
                                "main_falling": int(table.is_main_falling.sum()),
                                "sharp": int(main.morphology.str.startswith("sharp").sum()),
                                "gradual": int((main.morphology == "gradual").sum())})
comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("Udeco2 selected centers", "Udeco3 selected centers"))
for row, channel in enumerate((2, 3), start=1):
    edge = processed[f"Udeco{channel}"]["edge_signal"]
    fig.add_trace(go.Scattergl(x=time_s, y=edge, name=f"Udeco{channel}",
                               line=dict(color="#999", width=1)), row=row, col=1)
    for detector, color, symbol in (("median_diff", "#e41a1c", "circle"),
                                     ("haar", "#377eb8", "diamond")):
        table = comparison_events[detector][channel]
        main = table[table.is_main_rising | table.is_main_falling]
        centers = main.center_idx.to_numpy(dtype=int)
        fig.add_trace(go.Scattergl(x=time_s[centers], y=edge[centers], mode="markers",
                                   name=f"Udeco{channel} {detector}",
                                   marker=dict(color=color, symbol=symbol, size=10)), row=row, col=1)
if comparison_start_sec is not None and comparison_end_sec is not None:
    fig.update_xaxes(range=[comparison_start_sec, comparison_end_sec])
fig.update_xaxes(title_text="Time (s)", row=2, col=1)
fig.update_layout(title="Independent detector comparison on identical Uh ROIs",
                  template="plotly_white", height=800, dragmode="pan")
fig.show(config={"scrollZoom": True, "displaylogo": False})